# Stage 2 synthesis CV — true-noise mirror

10-fold (and optional 5-fold) synthesis CV with oracle **`noise_cat`**. Requires HP JSONs from [`abr_stage2_hp_tuning_true_noise.ipynb`](abr_stage2_hp_tuning_true_noise.ipynb).

No global Stage 1. Separate cache namespace (`stage2_synthesis_cv_true_*`). Mirror figures use `_true_noise` basenames. Pooled summary uses **`scenario` 1 / 2 / 3** (within / cross / combined), not train-slice A / B / C.

In [1]:
import json
from pathlib import Path

import pandas as pd

from utils.benchmark_metrics import (
    STAGE2_BEST_HP_TRUE_DIR,
    synthesis_cv_cache_paths,
)
from utils.stage2_hp import resolve_torch_device
from utils.stage2_synthesis_cv import (
    compute_pooled_synthesis_folds,
    panel_c_rmse_report_table,
    run_synthesis_cv,
    summarize_pooled_folds,
    summarize_pooled_oof_r2,
)
from utils.stage2_synthesis_plot import (
    synthesis_cohort_panels_3cv,
    synthesis_cohort_panels_3cv_r2,
)

device = resolve_torch_device()
print("device:", device)

device: mps


In [2]:
hp_by_scenario = {}
for scen in ("A", "B", "C"):
    p = STAGE2_BEST_HP_TRUE_DIR / f"{scen}.json"
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {p}; run HP tuning true-noise notebook first"
        )
    hp_by_scenario[scen] = json.loads(p.read_text(encoding="utf-8"))
    assert hp_by_scenario[scen].get("noise_label") == "true"
print("Loaded true HP for A/B/C")

Loaded true HP for A/B/C


In [3]:
n_folds = 10
paths = synthesis_cv_cache_paths(n_folds, variant="true")
folds_path, progress_path, summary_path = paths[:3]
pooled_folds, pooled_progress = paths[3], paths[4]
oof_path, _, pooled_oof_path, _, oof_r2_summary_path = (
    paths[5],
    paths[6],
    paths[7],
    paths[8],
    paths[9],
)

_, summary_10 = run_synthesis_cv(
    hp_by_scenario,
    noise_label="true",
    n_splits=n_folds,
    folds_path=folds_path,
    progress_path=progress_path,
    summary_path=summary_path,
    device=device,
    verbose=True,
)
pooled_folds_df = compute_pooled_synthesis_folds(
    hp_by_scenario,
    noise_label="true",
    n_splits=n_folds,
    folds_path=pooled_folds,
    progress_path=pooled_progress,
    device=device,
    verbose=True,
)
summary_pooled = summarize_pooled_folds(pooled_folds_df)
display(panel_c_rmse_report_table(summary_pooled))
summary_10_pooled = pd.concat([summary_10, summary_pooled], ignore_index=True)
# Pooled ``scenario`` 1/2/3 = within / cross / combined (not train-slice A/B/C).
display(summary_10_pooled.loc[summary_10_pooled["test_set"].eq("Pooled")])

Synthesis CV (true): 10-fold → stage2_synthesis_cv_true_summary.parquet
Pooled synthesis CV (true): 10-fold → stage2_synthesis_cv_true_pooled_folds.parquet


,test_set,scenario,model,RMSE,RMSE_sem,source,n_folds
0,Brad,all,ceiling,2.961628,0.149565,ceiling,10
1,Liberman,all,ceiling,2.341969,0.098753,ceiling,10
2,Brad,A,LR baseline,3.284766,0.114825,classical,10
3,Brad,A,RF,2.893845,0.135707,classical,10
4,Brad,A,XGB,2.760765,0.170304,classical,10


In [4]:
png3 = synthesis_cohort_panels_3cv(
    summary_10_pooled,
    fname="deck_act4_synthesis_cohorts_RMSE_cv_3panel_true_noise.png",
)
print("Wrote", png3)

Wrote /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_cohorts_RMSE_cv_3panel_true_noise.png


In [5]:
# OOF backfill (resume-friendly). OOF schema v2 dedupes rows and stays aligned with RMSE.
_, _ = run_synthesis_cv(
    hp_by_scenario,
    noise_label="true",
    n_splits=n_folds,
    folds_path=folds_path,
    progress_path=progress_path,
    summary_path=summary_path,
    device=device,
    verbose=True,
    force_rerun=False,
)
compute_pooled_synthesis_folds(
    hp_by_scenario,
    noise_label="true",
    n_splits=n_folds,
    folds_path=pooled_folds,
    progress_path=pooled_progress,
    device=device,
    verbose=True,
    force_rerun=False,
)
cohort_oof = pd.read_parquet(oof_path)
pooled_oof = pd.read_parquet(pooled_oof_path)
r2_summary_fig = summarize_pooled_oof_r2(cohort_oof, pooled_oof)
r2_summary_fig.to_parquet(oof_r2_summary_path, index=False)
display(r2_summary_fig.loc[r2_summary_fig["model"].eq("MLP")])
png_s2 = synthesis_cohort_panels_3cv_r2(
    r2_summary_fig,
    fname="supplementary_figure_S2_pooled_oof_r2_true_noise.png",
)
print("Wrote", png_s2, oof_r2_summary_path)

Synthesis CV (true): 10-fold → stage2_synthesis_cv_true_summary.parquet
Pooled synthesis CV (true): 10-fold → stage2_synthesis_cv_true_pooled_folds.parquet
Wrote /Users/nowaki027/MSDS/Practicum/figures/presentation/supplementary_figure_S2_pooled_oof_r2_true_noise.png /Users/nowaki027/MSDS/Practicum/figures/cache/stage2_synthesis_cv_true_oof_r2_summary.parquet
